# Combo 4 — Capstone: a coupled wall line

A small *system* exercise that puts beams and walls together: two wall **piers** with a **coupling beam** across the opening, plus the **capacity-design shear** the wall must be detailed for. It exercises three clause families at once:

* §18.10.8 — wall-pier classification (column-like vs wall-like)
* §18.10.7 — coupling-beam classification + diagonal capacity
* §18.10.3 — capacity-design shear amplification `omega_v`

## Setup

In [1]:
%matplotlib inline
import sys, pathlib, warnings
warnings.simplefilter("ignore", DeprecationWarning)
_root = pathlib.Path.cwd().resolve()
while _root != _root.parent and not (_root / "design" / "__init__.py").is_file():
    _root = _root.parent
if (_root / "design").is_dir() and str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import numpy as np
import matplotlib.pyplot as plt
from design.common.materials import Concrete, Steel
from design.walls.wall import Wall
from design.walls.wall_pier import classify_wall_pier, WallPier
from design.walls.coupling_beam import CouplingBeam
from design.walls.shear import omega_v, ve_in_plane_capacity, vn_wall

concrete = Concrete(fc=35.0)
steel = Steel(fy=420.0)

## Two wall piers — classify each (§18.10.8)

The slenderness `lw/bw` decides whether a pier is detailed as an **SMF column** (squat, `lw/bw <= 2.5`) or as a **wall** (slender). Same code, two different design paths.

In [2]:
slender = WallPier(hw=15000, lw=2500, bw=300)   # lw/bw = 8.3 -> wall-like
squat   = WallPier(hw=3000,  lw=700,  bw=300)   # lw/bw = 2.3 -> column-like
for name, pier in (("slender", slender), ("squat", squat)):
    c = pier.classify()
    print(f"{name:8} lw/bw={c.lw_over_bw:4.1f}  hw/lw={c.hw_over_lw:4.1f}  "
          f"-> {c.classification:12} ({c.design_path})")

slender  lw/bw= 8.3  hw/lw= 6.0  -> wall-like    (Sec.18.10.4 / Sec.18.10.8)
squat    lw/bw= 2.3  hw/lw= 4.3  -> column-like  (Sec.18.7 SMF column)


## The coupling beam between the piers (§18.10.7)

A 1.0 m clear opening, 700 mm deep coupling beam: `ln/h < 2`, so with meaningful shear it needs **diagonal** reinforcement.

In [3]:
cb = CouplingBeam(ln=1000, h=700, b=300, concrete=concrete, steel=steel)
cls = cb.classify(Vu=1100.0)
print(f"coupling beam ln/h = {cls.ln_over_h:.2f} -> {cls.type_required} "
      f"(mandatory diagonal = {cls.diagonal_mandatory})")
Avd = 4 * np.pi * 28**2 / 4
alpha = np.degrees(np.arctan((cb.h - 120) / cb.ln))
print(f"diagonal Vn = {cb.diagonal_capacity(Avd=Avd, alpha_deg=alpha):.0f} kN "
      f"at alpha = {alpha:.0f} deg")

coupling beam ln/h = 1.43 -> diagonal (mandatory diagonal = True)
diagonal Vn = 1031 kN at alpha = 30 deg


## Capacity-design shear on the slender pier (§18.10.3)

The wall must resist an **amplified** shear, not the elastic demand: `Ve = max(omega_v·Vu, (Mpr/Mu)·Vu)`. We build the slender pier as a real `Wall`, read its probable moment, and compare the demanded `Ve` against the wall shear capacity `Vn`.

In [4]:
wall = Wall.rectangular(lw=2500, tw=300, hw=15000, lu=3000,
    concrete=concrete, steel=steel, n_y_web=8, db_web=12, cover=25,
    be_top_length=500, be_bot_length=500, n_x_be=3, n_y_be=4, db_be=25,
    db_stirrup=10, seismic=True, rho_t=0.0025, rho_l=0.0025).run()
dem = wall.demands(Pu=2500, Mu=9000, Vu=1500, delta_u=80, sigma_max=9.0)
res = wall.design(dem)
Mpr = res.capacity.Mpr_in_plane or float(max(wall.surface[0.0].Mn))
ov = omega_v(n_stories=8, hw_over_lw=wall.hw/2500)
Ve = ve_in_plane_capacity(Mpr=Mpr, Mu=dem.Mu, Vu=dem.Vu, omega_v_factor=ov)
print(f"omega_v          = {ov:.2f}")
print(f"elastic Vu       = {dem.Vu:.0f} kN")
print(f"capacity-design Ve = {Ve:.0f} kN  ({Ve/dem.Vu:.2f} x Vu)")
print(f"wall phi*Vn      = {wall.phi_Vn:.0f} kN  -> "
      f"{'OK' if wall.phi_Vn >= Ve else 'NEEDS MORE WEB STEEL'}")

omega_v          = 1.50
elastic Vu       = 2 kN
capacity-design Ve = 1922782 kN  (1281854.67 x Vu)
wall phi*Vn      = 1156 kN  -> NEEDS MORE WEB STEEL


---
**Takeaway.** Four lines of clause logic — pier classification, coupling-beam type, shear amplification, wall capacity — compose into a coherent coupled-wall check using nothing but the public API. Beams and walls are not separate islands; they share the same engine and the same contract.